# Step 2.11: a run's metrics and a trade explained

This notebook is the check of step 2.11. Every run of `ff.backtest.run` now works out its metrics in the Rust engine and keeps them in its record, `run.json`: its trades' win rate, profit factor, and expectancy in money, ticks, and R; its days' profit and loss by trade date, worst day, CVaR, and Sharpe, Sortino, probabilistic, and deflated Sharpe ratios; its maximum drawdown and the longest spell under a peak; its profit and loss again at one more tick of slippage; breakdowns of its trades; and the backtest-expert skill's robustness summary, five scores of 20, a verdict, and red flags. `run.explain(trade_id=...)` replays one trade through the reference engine on the run's own records, prints every event with the rule behind it, and checks that the reference engine fills the trade as the Rust engine did.

The notebook runs the first backtest's strategy (step 2.2) on the window of `notebooks/checks/2.10.ipynb`, prints its metrics, and explains its first trade stopped out at a loss. Pick the `.venv` kernel and run all cells. The notebook reads the store in `FEFAT_DATA_DIR` and writes one run, and its engine cache, under `FEFAT_HOME`.

## The window

The product and the window `notebooks/02-first-backtest.ipynb` takes: MES when your store holds its 1-minute bars, and the last 23 sessions of the contract that was the front longest in the last 120 days, so that no roll falls inside the window.

Expected: one line such as `MES from databento: MESU2026, 23 sessions from 2026-08-21 to 2026-09-23`.

In [ ]:
import os
from datetime import time as clock
from datetime import timedelta

import polars as pl
from IPython.display import Pretty, display

import fefat as ff
from fefat.backtest import RunConfig
from fefat.backtest.explain import explain
from fefat.config import resolve_paths
from fefat.market.calendar import counted
from fefat.market.saved_instruments import instrument_table

paths = resolve_paths(os.environ)
instruments = instrument_table(paths.data_dir)
listed = ff.market.datasets().filter(pl.col("schema") == "ohlcv-1m")
if listed.is_empty():
    raise ValueError(
        f"Expected 1-minute bars of at least one product in {paths.data_dir}, got none: "
        "import your files first (docs/data-guide.md)."
    )
chosen = listed.sort(
    pl.col("root") != "MES", "contracts", descending=[False, True], maintain_order=True
).row(0, named=True)
ROOT, PROVIDER, LAST = chosen["root"], chosen["provider"], chosen["last_trade_date"]
daily = ff.market.load(
    ROOT, "1d", LAST - timedelta(days=120), LAST, fidelity="1m", provider=PROVIDER
).collect()
fronts = (
    daily.group_by("contract")
    .agg(sessions=pl.len(), last=pl.col("trade_date").max())
    .sort("sessions", "last", descending=True)
)
CONTRACT = fronts.item(0, "contract")
held = daily.filter(pl.col("contract") == CONTRACT).get_column("trade_date").sort().tail(23)
START, END = held.min(), held.max()
print(f"{ROOT} from {PROVIDER}: {CONTRACT}, {counted(held.len(), 'session')} from {START} to {END}")

## The first backtest and its metrics

The strategy of step 2.2 as `notebooks/checks/2.10.ipynb` writes it, filling on the 1-minute bars and paying the fees of `retail-default`. A ratio that cannot be worked out, such as a profit factor without a losing trade, shows as `none`.

Expected: the trades and their statistics, the days and their ratios, the drawdown, the profit and loss at one more tick of slippage, below the run's own, the robustness summary's five scores, and its verdict with the red flags that fire. A window of weeks always flags `short_test_period`, since the skill asks for five years. A warning names the fee table when its first schedule starts after the window (F-FEE-01).

In [ ]:
instrument = instruments.get(ROOT)
zone = instrument.session_calendar().hours.time_zone
local_close = (
    pl.from_epoch("ts_close", time_unit="ns")
    .dt.replace_time_zone("UTC")
    .dt.convert_time_zone(zone)
    .dt.time()
)
cross_up = (pl.col("close") > pl.col("mean")) & (
    pl.col("close").shift(1) <= pl.col("mean").shift(1)
)
in_hours = pl.col("local_close").is_between(clock(8, 30), clock(15, 0), closed="left")
first_backtest = ff.SignalStrategy(
    instrument=ROOT,
    timeframe="5m",
    features={"mean": pl.col("close").rolling_mean(20), "local_close": local_close},
    long_entry=cross_up & in_hours,
    short_entry=pl.lit(False),
    long_exit=pl.col("local_close") >= clock(15, 5),
    execution=ff.Execution(
        entry=ff.Market(), stop=ff.Stop(ticks=12), targets=[ff.Target(ticks=24)], sizing=ff.Fixed(1)
    ),
)
config = RunConfig(fidelity="1m", fee_table="retail-default", provider=PROVIDER)
run = ff.backtest.run(first_backtest, START, END, config)
METRICS = run.metrics


def shown(value: float | None, form: str) -> str:
    return "none" if value is None else format(value, form)


trades, days, drawdown = METRICS.trades, METRICS.daily, METRICS.drawdown
currency = instrument.currency
print(
    f"Trades: {trades.trades}, {trades.wins} won, win rate {shown(trades.win_rate, '.1%')}, "
    f"profit factor {shown(trades.profit_factor, '.2f')}"
)
print(
    f"Expectancy: {shown(trades.expectancy, '.2f')} {currency}, "
    f"{shown(trades.expectancy_ticks, '.2f')} ticks, {shown(trades.expectancy_r, '.2f')} R"
)
worst = days.worst_day
worst_text = "none" if worst is None else f"{worst.pnl} {currency} on {worst.trade_date}"
print(f"Days: {days.days}, worst {worst_text}, CVaR 95 % {shown(days.cvar_95, '.2f')} {currency}")
print(
    f"Sharpe {shown(days.sharpe, '.2f')}, Sortino {shown(days.sortino, '.2f')}, "
    f"probabilistic Sharpe {shown(days.probabilistic_sharpe, '.3f')}"
)
print(
    f"Maximum drawdown: {drawdown.max_drawdown} {currency}, {drawdown.max_drawdown_share:.2%} "
    f"of the capital, longest spell under a peak {drawdown.longest_drawdown_days} days"
)
print(f"P&L: {METRICS.pnl} {currency}, at one more tick {METRICS.pnl_at_plus_one_tick} {currency}")
robustness = METRICS.robustness
display(
    pl.DataFrame(
        {
            "dimension": [str(dimension.name) for dimension in robustness.dimensions],
            "score": [dimension.score for dimension in robustness.dimensions],
            "of": [dimension.max_score for dimension in robustness.dimensions],
        }
    )
)
flags = ", ".join(str(flag.id) for flag in robustness.red_flags) or "none"
print(f"Verdict: {robustness.verdict}, {robustness.total_score} of 100, red flags: {flags}")

## A losing trade explained

The run's first trade stopped out at a loss, replayed through the reference engine from the decision that placed its entry to its exit. Each line is an event: when, what happened, and the rule of `docs/spec/fill-semantics.md` behind it in brackets. The fills follow with their rules, liquidity, and slippage in ticks. On bars, the stop's trigger is the 1-minute bar that reached it, which the line below the stop's fill shows with its open, high, low, and close.

Expected: the entry placed and filled (F-MKT-03 on bars), its stop and target placed (F-OCO-02), the stop filled (F-STP-04) with its slippage of one tick, the bar that reached it, and the target cancelled (F-OCO-01). The notebook stops with an error if the two engines fill the trade differently.

In [ ]:
losing = run.trades.filter(pl.col("exit_reason") == "stop", pl.col("pnl") < 0)
if losing.is_empty():
    raise ValueError(
        f"Expected a trade stopped out at a loss from {START} to {END}, got none: choose another "
        "window."
    )
TRADE = losing.item(0, "entry_order")
EXPLANATION = explain(run, TRADE)
display(Pretty(EXPLANATION.text()))

## Your check

In a notebook, run `run.explain(trade_id=…)` for a losing trade: the printed sequence shows the stop trigger tick and the fill with its rule id and slippage.

The last code cell shows the same lines, as display output so that no price is printed as text, for the run's first trade stopped out at a loss: the stop's trigger is the bar line under its fill, and the fill line ends with its rule, `F-STP-04`, and its slippage in ticks.